# HBCC - chạy toàn bộ hai phiên không augmentation

Notebook này chạy tuần tự quy trình duy nhất của repository:

1. Phiên baseline: ResNet-18, MobileNetV2, ShuffleNetV2, CoC baseline, HBCC-Small và HBCC-Medium với CE.
2. Phiên KD: HBCC-Small và HBCC-Medium học từ ResNet-18 teacher cùng dataset/seed.

Mặc định notebook chạy cả CIFAR-10 và CIFAR-100: tổng cộng 16 run. Tất cả transform chỉ gồm ToTensor + Normalize.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
import yaml


def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'tools' / 'run_two_sessions.py').is_file():
            return candidate
    raise FileNotFoundError('Khong tim thay repository chua tools/run_two_sessions.py')


ROOT = find_repo_root()
print('Repository :', ROOT)
print('Python     :', sys.executable)
print('PyTorch    :', torch.__version__)
print('CUDA       :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))

## Cấu hình chạy

- Giữ DATASETS = ['cifar10', 'cifar100'] để chạy đầy đủ cả hai dataset.
- Chỉnh BASELINE_EPOCHS và KD_EPOCHS để đặt riêng số epoch cho phiên train thường và phiên knowledge distillation.
- Đặt SMOKE = True để kiểm tra nhanh bằng FakeData, mỗi run chỉ một batch.
- FORCE = False bảo vệ các run đã có; run hoàn tất và đúng metadata sẽ tự động được bỏ qua.

In [ ]:
DATASETS = ['cifar10', 'cifar100']
SEEDS = [42]
BASELINE_EPOCHS = 300
KD_EPOCHS = 250
DATA_ROOT = ROOT / 'data'
OUTPUT_ROOT = ROOT / 'runs_two_sessions'

SMOKE = False
FORCE = False
SHOW_PROGRESS = True

assert DATASETS and set(DATASETS) <= {'cifar10', 'cifar100'}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert isinstance(BASELINE_EPOCHS, int) and BASELINE_EPOCHS > 0
assert isinstance(KD_EPOCHS, int) and KD_EPOCHS > 0
print('Datasets    :', DATASETS)
print('Seeds       :', SEEDS)
print('CE epochs   :', BASELINE_EPOCHS)
print('KD epochs   :', KD_EPOCHS)
print('Data root   :', DATA_ROOT)
print('Output root :', OUTPUT_ROOT)
print('Smoke       :', SMOKE)

## Preflight

Kiểm tra recipe không augmentation, teacher ResNet-18, hai HBCC student và forward shape của toàn bộ model trước khi bắt đầu huấn luyện.

In [ ]:
RUNNER = ROOT / 'tools' / 'run_two_sessions.py'


def run_command(command: list[str]) -> None:
    print('\n>', subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


for dataset in DATASETS:
    run_command([
        sys.executable, str(RUNNER),
        '--dataset', dataset,
        '--baseline-epochs', str(BASELINE_EPOCHS),
        '--kd-epochs', str(KD_EPOCHS),
        '--validate-only',
    ])

## Chạy toàn bộ hai phiên

Runner luôn chạy baseline trước để tạo teacher checkpoint, sau đó mới chạy HBCC KD. Mỗi dataset/seed tạo 6 run CE và 2 run KD.

In [ ]:
for dataset in DATASETS:
    command = [
        sys.executable, str(RUNNER),
        '--dataset', dataset,
        '--data-root', str(DATA_ROOT),
        '--output', str(OUTPUT_ROOT),
        '--python', sys.executable,
        '--seeds', *[str(seed) for seed in SEEDS],
        '--baseline-epochs', str(BASELINE_EPOCHS),
        '--kd-epochs', str(KD_EPOCHS),
    ]
    if SHOW_PROGRESS:
        command.append('--progress')
    if SMOKE:
        command.append('--smoke')
    if FORCE:
        command.append('--force')
    run_command(command)

## Tổng hợp kết quả test

Cell này đọc config và test_metrics.json của từng run, kiểm tra đủ 8 kết quả cho mỗi dataset/seed rồi lưu một file CSV tổng hợp.

In [ ]:
records = []
selected_datasets = set(DATASETS)
selected_seeds = set(SEEDS)

for metrics_path in sorted(OUTPUT_ROOT.glob('*/test_metrics.json')):
    run_dir = metrics_path.parent
    config_path = run_dir / 'config.yaml'
    if not config_path.is_file():
        continue
    cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    dataset = cfg.get('protocol', {}).get('dataset')
    session = cfg.get('protocol', {}).get('session')
    seed = int(cfg.get('train', {}).get('seed', -1))
    run_epochs = int(cfg.get('train', {}).get('epochs', -1))
    is_smoke_run = cfg.get('data', {}).get('name') == 'fake'
    expected_epochs = 1 if SMOKE else (BASELINE_EPOCHS if session == 'baseline' else KD_EPOCHS)
    if dataset not in selected_datasets or seed not in selected_seeds:
        continue
    if session not in {'baseline', 'kd'} or run_epochs != expected_epochs or is_smoke_run != SMOKE:
        continue
    if session == 'kd':
        teacher_run = cfg.get('experiment', {}).get('teacher_run')
        teacher_config_path = OUTPUT_ROOT / str(teacher_run) / 'config.yaml'
        if not teacher_config_path.is_file():
            continue
        teacher_cfg = yaml.safe_load(teacher_config_path.read_text(encoding='utf-8'))
        expected_teacher_epochs = 1 if SMOKE else BASELINE_EPOCHS
        if int(teacher_cfg.get('train', {}).get('epochs', -1)) != expected_teacher_epochs:
            continue
    records.append({
        'dataset': dataset,
        'session': session,
        'model': cfg['experiment']['model_key'],
        'seed': seed,
        'epochs': run_epochs,
        'test_acc1': float(metrics['test_acc1']),
        'test_acc5': metrics.get('test_acc5'),
        'run': run_dir.name,
    })

summary = pd.DataFrame(records)
if summary.empty:
    raise RuntimeError('Khong tim thay ket qua test trong output root')

summary = summary.sort_values(['dataset', 'seed', 'session', 'model']).reset_index(drop=True)
expected = len(DATASETS) * len(SEEDS) * 8
if len(summary) != expected:
    raise RuntimeError(f'Ket qua chua du: tim thay {len(summary)}/{expected} run')

summary_path = OUTPUT_ROOT / 'two_sessions_summary.csv'
summary.to_csv(summary_path, index=False)
print('Da luu:', summary_path)
summary